# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shreyamT/flyrank_new/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Provisional Lane:** **Lane 2 — Refresh / Content Opportunity Scoring**

**Why this lane:**
Organic content decay is a critical operational bottleneck for multi-domain content portfolios. Over time, previously high-performing articles experience natural ranking degradation, shifts in search intent, and increased competitive pressure. However, editorial and SEO teams operate under strictly finite bandwidth: an editorial team can realistically audit, rewrite, or update only 20 to 50 URLs per week across an inventory of tens of thousands of published pages. Today, content updates are largely scheduled reactively (after organic clicks have completely collapsed) or arbitrarily (based on calendar age alone, which frequently targets pages that do not need attention).

Lane 2 directly addresses this human-in-the-loop bottleneck. Rather than treating ML as an abstract predictive exercise, this lane connects observable search and engagement performance signals to an operational decision: **generating an evidence-backed, prioritized review queue with transparent reason codes**. This ensures limited editorial hours are systematically allocated to the highest-leverage content assets where proactive intervention can preserve organic traffic and revenue before severe ranking collapse occurs.

In [1]:
import os, sys
from pathlib import Path
import pandas as pd
import numpy as np

# Locate repo root and starter CSV robustly
def find_data_path():
    candidates = [
        Path("data/raw/content_refresh_anonymized.csv"),
        Path("../data/raw/content_refresh_anonymized.csv"),
        Path("../../data/raw/content_refresh_anonymized.csv"),
    ]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError("Could not find content_refresh_anonymized.csv")

data_path = find_data_path()
print(f"Dataset located at: {data_path}")
print(f"Selected Lane: Lane 2 — Refresh / Content Opportunity Scoring")


Dataset located at: data/raw/content_refresh_anonymized.csv
Selected Lane: Lane 2 — Refresh / Content Opportunity Scoring


## 2. The question: decision, action, cost of a wrong call

**The Research Question:**
*Which declining, high-visibility content assets should editorial and SEO teams review and refresh first to prevent traffic loss and maximize recovery potential?*

**1. What decision does this work improve?**
The weekly allocation of editorial review capacity. Instead of arbitrarily picking pages by publication date or waiting for stakeholder complaints after traffic crashes, the team decides which specific 25–50 URLs to investigate and refresh in each operational sprint based on an empirical opportunity score.

**2. Who acts on the output, and what do they do?**
- **Actor:** Content editors, SEO specialists, and digital publishing strategists.
- **Action:** They pull the top-K ranked URLs from the review queue and perform targeted interventions based on the accompanying reason codes: updating outdated data/dates, expanding thin sections, re-optimizing title and meta descriptions for changed search intent, strengthening internal links, or consolidating cannibalized content.

**3. What does a wrong recommendation cost?**
- **False Positive (recommending a page that does not need refresh or is suffering temporary noise/seasonality):**
  Costs 4 to 8 hours of editorial research and rewriting time per page (~$200–$500 in human labor). Critically, making unnecessary edits to a well-ranking, stable page risks destabilizing its existing SERP ranking signals and degrading its search performance.
- **False Negative (failing to flag a genuinely decaying, high-value page):**
  The page drops off Page 1 of search results (falling from ranks 1–5 to rank 11+). Because organic click-through rates fall exponentially past the top positions, a missed decay leads to substantial, ongoing organic traffic and revenue loss that can take months or thousands of dollars in paid search to compensate for.

**4. Why data and ML help (why a plain rule isn't enough):**
A simple heuristic rule (such as `days_since_update > 180` or `traffic_drop > 20%`) fails in production. It produces thousands of false positives (flagging evergreen pages or seasonal dips) while missing nuanced decay patterns across multi-signal interactions—such as a page holding high impressions but experiencing gradual position slippage, declining CTR, and degrading on-page engagement. ML earns its place by synthesizing tangled non-linear signals across exposure, engagement, position, and freshness into a calibrated probability score that reliably ranks candidates by empirical risk and recovery leverage.

- **Unit of Analysis:** A single published content item (`content_id`) within a client domain (`client_id`) evaluated over a trailing 90-day performance window.
- **Task Type & Target:** Ranking / scoring and supervised classification predicting content decline risk (`trend_direction == 'down'`).
- **Success Metric:** Precision@K (specifically Precision@50 on a held-out test split of unseen clients) and ROC-AUC / Average Precision.

In [2]:
# Task framing specification summary
framing_spec = {
    "Unit of Analysis": "Single content item (content_id) within client_id (trailing 90-day window)",
    "Task Type": "Ranking & Scoring (Priority Queue) / Supervised Classification",
    "Candidate Target": "Decline risk proxy (trend_direction == 'down') evaluated on holdout clients",
    "Primary Evaluation Metric": "Precision@50 and ROC-AUC / Average Precision",
    "Decision Beneficiary": "Editorial & SEO Content Strategy Teams",
}

for key, val in framing_spec.items():
    print(f"{key:28s}: {val}")


Unit of Analysis            : Single content item (content_id) within client_id (trailing 90-day window)
Task Type                   : Ranking & Scoring (Priority Queue) / Supervised Classification
Candidate Target            : Decline risk proxy (trend_direction == 'down') evaluated on holdout clients
Primary Evaluation Metric   : Precision@50 and ROC-AUC / Average Precision
Decision Beneficiary        : Editorial & SEO Content Strategy Teams


## 3. Quick look at the data (2-3 real numbers)

To verify that Lane 2 is worth the next 7 weeks, we inspect the starter dataset (`data/raw/content_refresh_anonymized.csv`, containing 30,000 anonymized items across 32 clients) and evaluate the scale of content decay and human editorial capacity.

Below, we compute three key supporting metrics:
1. **Pervasiveness of content decay:** Over half of all content inventory is trending downward.
2. **High-value assets in active decline:** Thousands of prime Page 1 ranking pages with substantial search demand are actively losing momentum.
3. **The human bandwidth bottleneck:** Editorial capacity covers less than 1% of the decaying inventory, demonstrating why an intelligent ranking model is necessary.

In [3]:
# Load the starter dataset and compute supporting numbers
df = pd.read_csv(data_path)

total_rows = len(df)
unique_clients = df["client_id"].nunique()

# 1. Pervasiveness of content decay
declining_mask = df["trend_direction"] == "down"
num_declining = declining_mask.sum()
pct_declining = (num_declining / total_rows) * 100

# 2. High-value assets at risk (Page 1 ranking + high search demand)
# Recall from data dictionary: avg_position = 0 means 'no data', 0 < avg_position <= 10 means Page 1
page_one_mask = (df["avg_position"] > 0) & (df["avg_position"] <= 10)
num_page_one = page_one_mask.sum()
page_one_declining = (page_one_mask & declining_mask).sum()
pct_page_one_declining = (page_one_declining / num_page_one) * 100

# High demand on Page 1 (impressions_90d >= 500)
high_imp_mask = df["impressions_90d"] >= 500
page_one_high_demand_declining = (page_one_mask & high_imp_mask & declining_mask).sum()

# 3. Operational review capacity bottleneck
# Assuming an editorial team reviews 25 pages per week (~100 pages per month)
monthly_capacity = 100
coverage_of_declining = (monthly_capacity / num_declining) * 100
coverage_of_p1_high_demand = (monthly_capacity / page_one_high_demand_declining) * 100

print(f"=== STARTER DATASET EVIDENCE (Lane 2: Refresh Scoring) ===")
print(f"Total Content Items Analyzed : {total_rows:,} across {unique_clients} clients")
print(f"\n[Metric 1] Overall Content Decay Rate:")
print(f"  - Declining Items           : {num_declining:,} out of {total_rows:,} ({pct_declining:.2f}%)")
print(f"\n[Metric 2] High-Value Page 1 Assets at Risk:")
print(f"  - Total Page 1 Items (pos 1-10): {num_page_one:,}")
print(f"  - Page 1 Items Declining       : {page_one_declining:,} ({pct_page_one_declining:.2f}% of Page 1)")
print(f"  - Page 1 + High Demand (>=500 imp) Declining: {page_one_high_demand_declining:,}")
print(f"\n[Metric 3] Human Editorial Capacity Bottleneck:")
print(f"  - Monthly Team Capacity        : {monthly_capacity} reviews/month (25/week)")
print(f"  - Portfolio Decay Coverage     : {coverage_of_declining:.2f}% of all declining pages")
print(f"  - High-Demand Page 1 Coverage  : {coverage_of_p1_high_demand:.2f}% of high-impact decaying pages")


=== STARTER DATASET EVIDENCE (Lane 2: Refresh Scoring) ===
Total Content Items Analyzed : 30,000 across 32 clients

[Metric 1] Overall Content Decay Rate:
  - Declining Items           : 16,262 out of 30,000 (54.21%)

[Metric 2] High-Value Page 1 Assets at Risk:
  - Total Page 1 Items (pos 1-10): 12,983
  - Page 1 Items Declining       : 7,311 (56.31% of Page 1)
  - Page 1 + High Demand (>=500 imp) Declining: 4,454

[Metric 3] Human Editorial Capacity Bottleneck:
  - Monthly Team Capacity        : 100 reviews/month (25/week)
  - Portfolio Decay Coverage     : 0.61% of all declining pages
  - High-Demand Page 1 Coverage  : 2.25% of high-impact decaying pages


## 4. Careful words: what I can and can't claim

Honest scientific framing requires explicit boundaries between what our empirical analysis can substantiate and what it cannot.

### What I CAN claim:
- **Observational associations:** We can demonstrate measurable statistical relationships between observable features (content age, position stability, impression tiers, engagement rate, CTR gaps) and content decline.
- **Decision-support ranking:** We can provide a prioritized, empirical review queue that orders candidate pages by predicted decay probability and potential recovery impact, significantly improving upon random selection or naive single-variable sorting (e.g. sorting by age alone).
- **Quantifiable offline validation:** We can report empirical ranking metrics (such as Precision@50, ROC-AUC, and Average Precision) on holdout evaluation splits using unseen clients, providing clear benchmarks against transparent rule-based baselines.

### What I CANNOT claim:
- **Causal guarantees:** We **cannot** claim that updating or rewriting a page will *cause* Google to increase its rankings or restore organic impressions. Establishing causality requires controlled A/B testing or randomized editorial experiments, which purely observational data cannot provide.
- **Reverse-engineering Google's search algorithm:** We are not modeling or "predicting Google's algorithm." We are observing historical search and user behavior on published pages.
- **Deterministic recovery:** We cannot guarantee traffic recovery for any specific URL; individual search outcomes depend on external factors beyond the scope of this dataset, including competitors' actions, algorithm updates, and macro search interest shifts.

In [4]:
# Verification of observable signals vs product decisions
# Ensure we rely strictly on observable signals, avoiding circular product flags or leakage
prohibited_features = ["trend_direction", "trend_pct"]  # These define the label; never use as features
leakage_check = [col for col in prohibited_features if col in df.columns]
print(f"Dataset columns checked: {len(df.columns)} total observable and derived fields.")
print(f"Label-defining fields identified and quarantined: {leakage_check} (NEVER to be used as input features)")
print("Observable signals ready for honest baseline and feature construction.")


Dataset columns checked: 44 total observable and derived fields.
Label-defining fields identified and quarantined: ['trend_direction', 'trend_pct'] (NEVER to be used as input features)
Observable signals ready for honest baseline and feature construction.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.